<a href="https://colab.research.google.com/github/juanpablor69/Proyecto_IA/blob/main/05_XGBOOST_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto - Inteligencia Artificial para las Ciencias y las Ingenierías**
## Notebook 05 usando XGBOOST
Accuracy XGBoost: 0.4340361010830325 | Score Kaggle: 0.36393

Autor: Juan Pablo Rendón Jimenez. \
Universidad de Antioquia

El **objetivo** de este notebook será aproximarnos a la solucion final usando distintos modelos junto a estrategias de preprocesado.






## Enlace con Kaggle

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '.'
!chmod 600 ./kaggle.json
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 1.63GB/s]


## Lectura e inspeccion de datos

In [ ]:
!unzip udea*.zip > /dev/null
!wc *.csv

   296787    296787   4716673 submission_example.csv
   296787   4565553  59185238 test.csv
   692501  10666231 143732437 train.csv
  1286075  15528571 207634348 total


In [ ]:
import pandas as pd
import numpy as np

datos = pd.read_csv("train.csv")
print("Dimensiones nodel dataset:", datos.shape)

Dimensiones del dataset: (692500, 21)


La base de datos contiene 692500 filas y 21 columnas. La estructura es la siguiente:

In [ ]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


## Analisis de la base de datos:

In [ ]:
print(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ID                           692500 non-null  int64  
 1   PERIODO_ACADEMICO            692500 non-null  int64  
 2   E_PRGM_ACADEMICO             692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD  686213 non-null  object 
 5   E_HORASSEMANATRABAJA         661643 non-null  object 
 6   F_ESTRATOVIVIENDA            660363 non-null  object 
 7   F_TIENEINTERNET              665871 non-null  object 
 8   F_EDUCACIONPADRE             669322 non-null  object 
 9   F_TIENELAVADORA              652727 non-null  object 
 10  F_TIENEAUTOMOVIL             648877 non-null  object 
 11  E_PRIVADO_LIBERTAD           692500 non-null  object 
 12  E_PAGOMATRICULAPROPIO        686002 non-null  object 
 13 

In [ ]:
print("\n--- Valores faltantes por columna ---")
print(datos.isnull().sum())


--- Valores faltantes por columna ---
ID                                 0
PERIODO_ACADEMICO                  0
E_PRGM_ACADEMICO                   0
E_PRGM_DEPARTAMENTO                0
E_VALORMATRICULAUNIVERSIDAD     6287
E_HORASSEMANATRABAJA           30857
F_ESTRATOVIVIENDA              32137
F_TIENEINTERNET                26629
F_EDUCACIONPADRE               23178
F_TIENELAVADORA                39773
F_TIENEAUTOMOVIL               43623
E_PRIVADO_LIBERTAD                 0
E_PAGOMATRICULAPROPIO           6498
F_TIENECOMPUTADOR              38103
F_TIENEINTERNET.1              26629
F_EDUCACIONMADRE               23664
RENDIMIENTO_GLOBAL                 0
INDICADOR_1                        0
INDICADOR_2                        0
INDICADOR_3                        0
INDICADOR_4                        0
dtype: int64


## Limpieza de datos
Luego del analisis realizado identificamos dos columnas completamente identicas (Internet), por lo tanto procederemos con su eliminacion. Ademas, observamos un alto numero de valores faltantes, por lo que rellenaremos esos espacios con un NO REPORTA para evitar errores o sesgos.

In [ ]:
def limpiar_df(df):
    df = df.copy()

    # 1. Eliminar columnas duplicadas como F_TIENEINTERNET.1
    duplicadas = [c for c in df.columns if ".1" in c]
    df = df.drop(columns=duplicadas, errors="ignore")

    # 2. Convertir categorías tipo object a category
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype("category")

    return df


In [ ]:
#Eliminar columna duplicada
if 'F_TIENEINTERNET.1' in datos.columns:
  datos.drop(columns=['F_TIENEINTERNET.1'], inplace=True)

In [ ]:
datos

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,Si,N,No,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,No,N,No,Si,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,No,No,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,No,N,No,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,Si,N,No,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692495,25096,20195,BIOLOGIA,LA GUAJIRA,Entre 500 mil y menos de 1 millón,Entre 11 y 20 horas,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,Si,Si,Secundaria (Bachillerato) incompleta,medio-alto,0.237,0.271,0.271,0.311
692496,754213,20212,PSICOLOGIA,NORTE SANTANDER,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Primaria incompleta,Si,No,N,No,Si,Secundaria (Bachillerato) incompleta,bajo,0.314,0.240,0.278,0.260
692497,504185,20183,ADMINISTRACIÓN EN SALUD OCUPACIONAL,BOGOTÁ,Entre 1 millón y menos de 2.5 millones,Menos de 10 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,Si,Si,Secundaria (Bachillerato) incompleta,medio-bajo,0.286,0.240,0.314,0.287
692498,986620,20195,PSICOLOGIA,TOLIMA,Entre 2.5 millones y menos de 4 millones,Menos de 10 horas,Estrato 1,No,Primaria completa,No,No,N,Si,Si,Primaria completa,bajo,0.132,0.426,0.261,0.328


In [ ]:
# 3. Reemplazos especiales
datos['F_EDUCACIONMADRE'] = datos['F_EDUCACIONMADRE'].replace(
    ['No sabe', 'No Aplica'], 'no info'
)

# 4. Mapear columna ordinal
mapa_matricula = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0,
    'no info': -1
}
datos['E_VALORMATRICULAUNIVERSIDAD'] = datos['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)

# 5. Imputación numérica
columnas_numericas = datos.select_dtypes(include=[np.number]).columns
datos[columnas_numericas] = datos[columnas_numericas].fillna(0)

In [ ]:
y = datos["RENDIMIENTO_GLOBAL"]
X = datos.drop(columns=["RENDIMIENTO_GLOBAL"])


In [ ]:
cat_cols = X.select_dtypes(include="object").columns
cat_cols

Index(['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_HORASSEMANATRABAJA',
       'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
       'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD',
       'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'],
      dtype='object')

In [ ]:
#datos = pd.get_dummies(datos, columns=columnas_categoricas, drop_first=True)

## Conversion de columnas en one-hot

En la entrega 3 usaremos modelos basados en Random Forest, entre otros. Para hacer uso de estos modelos necesitaremos convertir algunos datos ya que ellos solo trabajan con valores numéricos. En este caso vamos a reemplazar los datos como "Si" o "No" por una variable booleana como 0 y 1.

In [ ]:

#for col in datos.select_dtypes(include=['int64']).columns:
 #   datos[col] = datos[col].astype('int32')

#for col in datos.select_dtypes(include=['float64']).columns:
 #   datos[col] = datos[col].astype('float32')


--

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train = limpiar_df(X_train)

In [ ]:
le_y = LabelEncoder()
y_train_enc = le_y.fit_transform(y_train)
y_val_enc   = le_y.transform(y_val)


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist"   # 🔥 usa menos RAM
)

xgb_model.fit(X_train, y_train_enc)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_xgb = xgb_model.predict(X_val)
print("Accuracy XGBoost:", accuracy_score(y_val_enc, y_pred_xgb))


Accuracy XGBoost: 0.4340361010830325


Subir a Kaggle

In [ ]:
# 1. Cargar el test
test_data = pd.read_csv("test.csv")

X_test_final = test_data.copy()
X_test_final = limpiar_df(X_test_final)
# Guardar IDs
zt_ids = X_test_final["ID"]

In [ ]:
# Aplicar LABEL ENCODERS conocidos
for col, le in label_encoders.items():
    X_test_final[col] = X_test_final[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

In [ ]:
# FIX: Convertir cualquier columna object restante
for col in X_test_final.select_dtypes(include=['object']).columns:
    X_test_final[col] = X_test_final[col].astype('category').cat.codes

In [ ]:
# Predicciones
preds_test_data = xgb_model.predict(X_test_final)

In [ ]:
preds_text = le_y.inverse_transform(preds_test_data)

In [ ]:
submission = pd.DataFrame({
    'ID': zt_ids,
    'RENDIMIENTO_GLOBAL': preds_text
})



In [ ]:
submission

,ID,RENDIMIENTO_GLOBAL
0,550236,bajo
1,98545,medio-alto
2,499179,alto
3,782980,bajo
4,785185,bajo
...,...,...
296781,496981,bajo
296782,209415,alto
296783,239074,alto
296784,963852,alto


In [ ]:
submission.to_csv("my_submission.csv", index=False)

!head my_submission.csv


ID,RENDIMIENTO_GLOBAL
550236,bajo
98545,medio-alto
499179,alto
782980,bajo
785185,bajo
58495,medio-bajo
705444,medio-alto
557548,alto
519909,medio-bajo


In [ ]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f my_submission.csv -m "submission XGBoost Notebook 99"


100% 3.93M/3.93M [00:02<00:00, 1.68MB/s]
Successfully submitted to UDEA/ai4eng 20252 - Pruebas Saber Pro Colombia